# Isolated Ablation: N1 / N3 Double DQN
Use the project's shared `.venv` and read-only `config/experiment.json` with a separate notebook kernel. Workflow: freeze -> local preflight -> submit -> wait -> download -> verify. Query existing calls without repeating training.
The default phase opens the registered first-pair pilot. Later phases require local verification of the preceding phase and explicit selection below. Studies L/Q/E/W are not implemented. Predicted risk is not audited true risk; this notebook does not claim a first-acceptance-v1 pass.

In [6]:
import sys
from pathlib import Path

cwd = Path.cwd().resolve()
ROOT = next((p for p in (cwd, *cwd.parents) if (p / 'config/experiment.json').is_file() and (p / 'ablations').is_dir()), None)
if ROOT is None:
    raise RuntimeError('Open this notebook from the project or ablations directory')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

In [7]:
import cloud_edge_rl
from ablations.cloud import DEFAULT_NAME, prepare_cloud
from ablations.runner import preflight

if not Path(cloud_edge_rl.__file__).resolve().is_relative_to(ROOT / 'src'):
    raise RuntimeError('Select this project .venv kernel; a different installation was imported')
# 执行选择，不是另一份科学 CONFIG。/ Execution selection, not a second scientific CONFIG.
PHASE = 'continue'  # pilot: 500+500；continue: 首对各续700+验证；replicate: 其余两对
RESULTS = {}
print({'project': str(ROOT), 'python': sys.executable, 'shared_config': str(ROOT / 'config/experiment.json'), 'phase': PHASE})

{'project': 'C:\\DevSpace\\PyDevspace\\RL\\RL_project', 'python': 'c:\\DevSpace\\PyDevspace\\RL\\RL_project\\.venv\\Scripts\\python.exe', 'shared_config': 'C:\\DevSpace\\PyDevspace\\RL\\RL_project\\config\\experiment.json', 'phase': 'continue'}


## 1. Open the Frozen Configuration and Risk Input
The only algorithmic difference is `n_step=1/3`. Each arm of the first pair starts with 500 interactions, then continues for 700 to reach 1,200, including warmup. All three pairs require at most 7,200 training and 180 validation interactions. Both arms are accounted for jointly per method/seed, with a conservative allocation of 90 shared collection interactions. Scientific configuration or source drift blocks submission.

In [8]:
session = prepare_cloud(ROOT)
print(session.status(PHASE))

{'experiment_id': 'ablation-n-7e21a45eb4e542529462c361acf7dc7a', 'phase': 'continue'}


## 2. Targeted Local Preflight
A 90-second process-tree watchdog bounds the preflight. Synthetic environments check N1 parity with the existing implementation, N3 terminal targets, restoration, and artifact contracts. No Adult data or cloud services are accessed. Following project conventions, tests reside in the ignored `ablations/test/` directory; missing tests explicitly block execution.

In [9]:
RESULTS['preflight'] = preflight(ROOT, name=DEFAULT_NAME)
print(RESULTS['preflight'])

{'status': 'local_synthetic_preflight_passed', 'process': {'pid': 28512, 'exit_code': 0, 'timed_out': False, 'elapsed_seconds': 53.39100000000326, 'process_tree_clean': True, 'surviving_pids': [], 'log_path': 'C:\\DevSpace\\PyDevspace\\RL\\RL_project\\ablations\\output\\N-cloud-v1\\controller\\preflight-d4ea9cbdcbc64eefafc968e4d8920afe.log'}, 'new_real_interactions': 0, 'new_cloud_calls': 0, 'live_modal_validated': False, 'execution_ready': False, 'official_test_access': False, 'formal_acceptance': 'not_evaluated', 'basis': 'Prior full preflight plus controller/actual-watchdog regression', 'original_preflight': 'C:\\DevSpace\\PyDevspace\\RL\\RL_project\\ablations\\output\\N-cloud-v1\\recovery\\cpu-verification-v1\\original-preflight.json', 'incremental_validation': 'C:\\Users\\42155\\AppData\\Local\\Temp\\opencode\\rl-ablation-cpu-verification-tests-20261010-v1.json', 'source_rebind_decision': 'C:\\DevSpace\\PyDevspace\\RL\\RL_project\\ablations\\output\\N-cloud-v1\\recovery\\cpu-verif

## 3. Submit the Selected Phase
Running this cell submits the selected phase; an existing call returns its original record. Selecting `continue` or `replicate` explicitly approves that phase without automatic expansion. Restarting cannot reset failed or unknown consumption. The first-pair pilot authorization and historical accounting must already be registered.

In [10]:
if PHASE != 'pilot':
    RESULTS['phase_approval'] = session.approve_phase(PHASE)
RESULTS['submission'] = session.submit(PHASE)
print(RESULTS['submission'])

{'experiment_id': 'ablation-n-7e21a45eb4e542529462c361acf7dc7a', 'phase': 'continue', 'attempt_id': 'ef5969da12924f3d9d5d55cfbe7e1829', 'status': 'submitted', 'call_id': 'fc-01M4KP25QJ3FAHER42AZEW5A4J', 'app_id': 'ap-fHo183WBniQObNezaJ3auk', 'volume_name': 'ablation-n-7e21a45eb4e542529462c361acf7dc7a', 'environment': 'main', 'reservation': {'schema': 'ablation_execution_v1', 'experiment_id': 'ablation-n-7e21a45eb4e542529462c361acf7dc7a', 'kind': 'reserve', 'phase': 'continue', 'payload': {'full_workflow_jobs': 18, 'cloud_limits': {'gpu': 'H100', 'cpu': 16, 'memory': 65536, 'max_containers': 1, 'retries': 0, 'deadline_seconds': 9000}, 'jobs': [{'arm': 'N1', 'seed': 96101, 'partition': 'train', 'interactions': 700}, {'arm': 'N1', 'seed': 96201, 'partition': 'validation', 'interactions': 30}, {'arm': 'N3', 'seed': 96101, 'partition': 'train', 'interactions': 700}, {'arm': 'N3', 'seed': 96201, 'partition': 'validation', 'interactions': 30}], 'planned_jobs': [{'phase': 'pilot', 'arm': 'N1',

## 4. Wait for the Original Call
Explicit timeouts and a single-container limit apply. After completion, the following cell downloads the results. Closing the notebook does not cancel a submitted detached task. After restarting the kernel, reopen the same record from step 1. To stop early, run `session.cancel(PHASE)` in a separate cell.

In [11]:
RESULTS['status'] = session.wait(PHASE)
print(RESULTS['status'])

{'status': 'cloud_verified', 'result': {'experiment_id': 'ablation-n-7e21a45eb4e542529462c361acf7dc7a', 'phase': 'continue', 'attempt_id': 'ef5969da12924f3d9d5d55cfbe7e1829', 'status': 'cloud_verified', 'actual_cloud_cost_usd': None, 'formal_acceptance': False, 'risk_measurement': 'predicted_risk', 'nodes': {'ablation-n-7e21a45eb4e542529462c361acf7dc7a-N1-96101-train700': {'status': 'verified', 'result': {'run_id': 'ablation-n-7e21a45eb4e542529462c361acf7dc7a-N1-96101-train700', 'stage': 'train_policy', 'risk_version': 'fresh-seven-methods-rerun-20261010-v1-risk-fit8401-v1', 'ablation_experiment_id': 'ablation-n-7e21a45eb4e542529462c361acf7dc7a', 'risk_measurement': 'predicted_risk', 'trained': True, 'new_environment_steps': 700, 'environment_steps': 1200, 'learned_steps': 720, 'warmup_steps': 480, 'updates': 465}, 'compute_process': {'pid': 19, 'exit_code': 0, 'timed_out': False, 'elapsed_seconds': 25.40965899, 'process_tree_clean': True, 'surviving_pids': [], 'log_path': '/tmp/ablati

## 5. Collect All Phase Artifacts
Download to an isolated phase directory with a two-hour limit. Archives and individual files retain completion receipts, so repeated execution reuses completed transfers. Pending or blocked states never launch new training.

In [13]:
if RESULTS['status'].get('status') == 'cloud_verified':
    RESULTS['transfer'] = session.collect(PHASE)
else:
    RESULTS['transfer'] = {'status': 'pending_or_blocked', 'transport_complete': False}
print(RESULTS['transfer'])

{'experiment_id': 'ablation-n-7e21a45eb4e542529462c361acf7dc7a-continue', 'transport_complete': True, 'status': 'transport_complete', 'graph_status': 'cloud_verified', 'semantic_verified': False, 'files': 11129, 'bytes': 686144100, 'destination': 'C:\\DevSpace\\PyDevspace\\RL\\RL_project\\ablations\\output\\N-cloud-v1\\phases\\continue\\artifacts', 'inventory': 'C:\\DevSpace\\PyDevspace\\RL\\RL_project\\ablations\\output\\N-cloud-v1\\phases\\continue\\artifacts\\experiments\\ablation-n-7e21a45eb4e542529462c361acf7dc7a-continue\\inventory.json', 'elapsed_seconds': 46.844000000011874, 'phase': 'continue', 'attempt_id': 'ef5969da12924f3d9d5d55cfbe7e1829'}


## 6. Verify Locally and Inspect Results
Check each node's configuration, weights, restoration state, warmup/learning counters, and cloud proof. A successful execution lifecycle does not establish algorithmic efficacy; the 500-interaction pilot is insufficient to conclude that the policy or true risk improved.

In [14]:
if RESULTS['transfer'].get('transport_complete') is True:
    RESULTS['verification'] = session.verify(PHASE)
    RESULTS['summary'] = session.summary(PHASE)
    print(RESULTS['summary'])
else:
    print('Waiting for a complete transfer; no semantic pass claimed')

{'phase': 'continue', 'status': 'verified', 'results': [{'phase': 'pilot', 'arm': 'N1', 'seed': 96101, 'run_id': 'ablation-n-7e21a45eb4e542529462c361acf7dc7a-N1-96101-train500', 'stage': 'train_policy', 'risk_version': 'fresh-seven-methods-rerun-20261010-v1-risk-fit8401-v1', 'ablation_experiment_id': 'ablation-n-7e21a45eb4e542529462c361acf7dc7a', 'risk_measurement': 'predicted_risk', 'trained': True, 'new_environment_steps': 500, 'environment_steps': 500, 'learned_steps': 296, 'warmup_steps': 204, 'updates': 41}, {'phase': 'pilot', 'arm': 'N3', 'seed': 96101, 'run_id': 'ablation-n-7e21a45eb4e542529462c361acf7dc7a-N3-96101-train500', 'stage': 'train_policy', 'risk_version': 'fresh-seven-methods-rerun-20261010-v1-risk-fit8401-v1', 'ablation_experiment_id': 'ablation-n-7e21a45eb4e542529462c361acf7dc7a', 'risk_measurement': 'predicted_risk', 'trained': True, 'new_environment_steps': 500, 'environment_steps': 500, 'learned_steps': 296, 'warmup_steps': 204, 'updates': 41}, {'phase': 'continu